# Velocity and Acceleration Analysis

This notebook demonstrates how to compute velocities and accelerations of all
joints in a four-bar linkage using pylinkage's numba-optimized solver. We
validate the analytical solver results against finite-difference approximations
to confirm correctness.

**Outline:**
1. Build a four-bar linkage from link lengths
2. Convert to solver data and run position-only simulation
3. Run full kinematic simulation (positions + velocities + accelerations)
4. Plot the coupler joint trajectory, velocity, and acceleration
5. Validate velocity against finite differences of position
6. Validate acceleration against finite differences of velocity
7. Compute peak velocity and acceleration for dynamic force analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pylinkage.solver import linkage_to_solver_data, simulate, simulate_with_kinematics
from pylinkage.synthesis import fourbar_from_lengths
from pylinkage.visualizer import plot_static_linkage

## 1. Build a Four-Bar Linkage

We create a Grashof crank-rocker with the following link lengths:
- Crank (a) = 1.0
- Coupler (b) = 3.0
- Rocker (c) = 3.0
- Ground (d) = 4.0

This satisfies the Grashof condition (s + l <= p + q) so the crank can
make a full rotation.

In [ ]:
iterations = 500

linkage = fourbar_from_lengths(
    crank_length=1.0,
    coupler_length=3.0,
    rocker_length=3.0,
    ground_length=4.0,
    iterations=iterations,
)

print(f"Joints: {[j.name for j in linkage.components]}")
print(f"Solve order: {[j.name for j in linkage._find_solve_order()]}")

## 2. Convert to Solver Data

The solver operates on contiguous numpy arrays for numba JIT performance.
`linkage_to_solver_data()` extracts positions, constraints, topology, and
solve order into a `SolverData` dataclass.

In [ ]:
data = linkage_to_solver_data(linkage)

print(f"Number of joints: {data.n_joints}")
print(f"Number of constraints: {data.n_constraints}")
print(f"Joint types: {data.joint_types}")
print(f"Solve order: {data.solve_order}")
print(f"Initial positions:\n{data.positions}")

## 3. Position-Only Simulation

First, run a basic position simulation using `simulate()`. This returns
a trajectory array of shape `(iterations, n_joints, 2)`.

In [ ]:
# We need a fresh copy since simulate modifies positions in place
data_pos = data.copy()

dt = 1.0  # Time step (matches the angle_step stored in constraints)

trajectory = simulate(
    data_pos.positions,
    data_pos.constraints,
    data_pos.joint_types,
    data_pos.parent_indices,
    data_pos.constraint_offsets,
    data_pos.solve_order,
    iterations=iterations,
    dt=dt,
)

print(f"Trajectory shape: {trajectory.shape}")
print(f"Any NaN values: {np.any(np.isnan(trajectory))}")

## 4. Plot Coupler Joint Trajectory

Joint C (index 3) is the coupler-rocker connection. Its path shows the
characteristic coupler curve of the four-bar.

In [ ]:
# Joint indices: 0=A (static), 1=D (static), 2=B (crank), 3=C (coupler)
coupler_idx = 3

cx = trajectory[:, coupler_idx, 0]
cy = trajectory[:, coupler_idx, 1]

# Convert solver trajectory array (iterations, n_joints, 2) to loci format
# expected by plot_static_linkage: list of tuples [(x1,y1), (x2,y2), ...]
loci_for_plot = [
    tuple((trajectory[i, j, 0], trajectory[i, j, 1]) for j in range(trajectory.shape[1]))
    for i in range(trajectory.shape[0])
]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(cx, cy, 'b-', linewidth=1.5, label='Coupler C trajectory')
ax.plot(cx[0], cy[0], 'go', markersize=8, label='Start')

# Draw the mechanism at its initial position
plot_static_linkage(linkage, ax, loci_for_plot, show_legend=True, show_loci=False,
                    title='Four-bar mechanism and coupler joint C trajectory')

plt.tight_layout()
plt.show()

## 5. Full Kinematic Simulation

Now we use `simulate_with_kinematics()` to compute positions, velocities,
and accelerations simultaneously. This requires providing angular velocity
and angular acceleration arrays for crank joints.

In [ ]:
from pylinkage.solver import JOINT_CRANK

data_kin = data.copy()

n_joints = data_kin.n_joints
velocities = np.zeros((n_joints, 2), dtype=np.float64)
accelerations = np.zeros((n_joints, 2), dtype=np.float64)

# Identify crank joints and their angular velocities
# The crank's angle_rate is stored in constraints[offset+1]
# For a constant-speed crank: omega = angle_rate / dt, alpha = 0
crank_indices_list = []
omega_list = []
for i in range(n_joints):
    if data_kin.joint_types[i] == JOINT_CRANK:
        crank_indices_list.append(i)
        offset = data_kin.constraint_offsets[i]
        angle_rate = data_kin.constraints[offset + 1]  # rad per step
        omega_list.append(angle_rate / dt)

crank_indices = np.array(crank_indices_list, dtype=np.int32)
omega_values = np.array(omega_list, dtype=np.float64)
alpha_values = np.zeros_like(omega_values)  # Constant speed: zero angular acceleration

print(f"Crank indices: {crank_indices}")
print(f"Angular velocities (rad/step): {omega_values}")
print(f"Angular accelerations (rad/step^2): {alpha_values}")

In [ ]:
pos_traj, vel_traj, acc_traj = simulate_with_kinematics(
    data_kin.positions,
    velocities,
    accelerations,
    data_kin.constraints,
    data_kin.joint_types,
    data_kin.parent_indices,
    data_kin.constraint_offsets,
    data_kin.solve_order,
    omega_values,
    alpha_values,
    crank_indices,
    iterations=iterations,
    dt=dt,
)

print(f"Position trajectory shape: {pos_traj.shape}")
print(f"Velocity trajectory shape: {vel_traj.shape}")
print(f"Acceleration trajectory shape: {acc_traj.shape}")

## 6. Plot Velocity Components

We plot the X and Y velocity components and the magnitude for the coupler
joint over one full crank rotation.

In [ ]:
time = np.arange(iterations) * dt
crank_angle_deg = np.degrees(omega_values[0] * time)

vx = vel_traj[:, coupler_idx, 0]
vy = vel_traj[:, coupler_idx, 1]
v_mag = np.sqrt(vx**2 + vy**2)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(crank_angle_deg, vx, 'r-', linewidth=1.5, label='vx')
axes[0].plot(crank_angle_deg, vy, 'b-', linewidth=1.5, label='vy')
axes[0].set_ylabel('Velocity')
axes[0].set_title('Coupler joint C: velocity components')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(crank_angle_deg, v_mag, 'k-', linewidth=2)
axes[1].set_xlabel('Crank angle (degrees)')
axes[1].set_ylabel('Speed (magnitude)')
axes[1].set_title('Coupler joint C: speed')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Plot Acceleration Components

In [ ]:
ax_c = acc_traj[:, coupler_idx, 0]
ay_c = acc_traj[:, coupler_idx, 1]
a_mag = np.sqrt(ax_c**2 + ay_c**2)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(crank_angle_deg, ax_c, 'r-', linewidth=1.5, label='ax')
axes[0].plot(crank_angle_deg, ay_c, 'b-', linewidth=1.5, label='ay')
axes[0].set_ylabel('Acceleration')
axes[0].set_title('Coupler joint C: acceleration components')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(crank_angle_deg, a_mag, 'k-', linewidth=2)
axes[1].set_xlabel('Crank angle (degrees)')
axes[1].set_ylabel('Acceleration (magnitude)')
axes[1].set_title('Coupler joint C: acceleration magnitude')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Validation: Velocity vs Finite Differences

We compute numerical velocity by taking the gradient of the position data
with `numpy.gradient`. If the analytical solver is correct, the two curves
should overlap closely.

In [ ]:
# Numerical velocity via central finite differences
cx_kin = pos_traj[:, coupler_idx, 0]
cy_kin = pos_traj[:, coupler_idx, 1]

vx_num = np.gradient(cx_kin, dt)
vy_num = np.gradient(cy_kin, dt)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(crank_angle_deg, vx, 'r-', linewidth=2, label='Solver vx')
axes[0].plot(crank_angle_deg, vx_num, 'k--', linewidth=1.5, label='Finite diff vx')
axes[0].set_ylabel('Velocity X')
axes[0].set_title('Velocity validation: solver vs finite differences')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(crank_angle_deg, vy, 'b-', linewidth=2, label='Solver vy')
axes[1].plot(crank_angle_deg, vy_num, 'k--', linewidth=1.5, label='Finite diff vy')
axes[1].set_xlabel('Crank angle (degrees)')
axes[1].set_ylabel('Velocity Y')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify the difference
# Skip first and last few points where boundary effects affect the gradient
margin = 5
vx_err = np.abs(vx[margin:-margin] - vx_num[margin:-margin])
vy_err = np.abs(vy[margin:-margin] - vy_num[margin:-margin])
print(f"Max |vx_solver - vx_numerical|: {np.max(vx_err):.6f}")
print(f"Max |vy_solver - vy_numerical|: {np.max(vy_err):.6f}")
print(f"Mean velocity error: {np.mean(np.sqrt(vx_err**2 + vy_err**2)):.6f}")

## 9. Validation: Acceleration vs Finite Differences

Similarly, we compute numerical acceleration from the solver's velocity
output using `numpy.gradient`, and compare with the analytical acceleration.

In [ ]:
# Numerical acceleration: derivative of solver velocity
ax_num = np.gradient(vx, dt)
ay_num = np.gradient(vy, dt)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(crank_angle_deg, ax_c, 'r-', linewidth=2, label='Solver ax')
axes[0].plot(crank_angle_deg, ax_num, 'k--', linewidth=1.5, label='Finite diff ax')
axes[0].set_ylabel('Acceleration X')
axes[0].set_title('Acceleration validation: solver vs finite differences')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(crank_angle_deg, ay_c, 'b-', linewidth=2, label='Solver ay')
axes[1].plot(crank_angle_deg, ay_num, 'k--', linewidth=1.5, label='Finite diff ay')
axes[1].set_xlabel('Crank angle (degrees)')
axes[1].set_ylabel('Acceleration Y')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify the difference
ax_err = np.abs(ax_c[margin:-margin] - ax_num[margin:-margin])
ay_err = np.abs(ay_c[margin:-margin] - ay_num[margin:-margin])
print(f"Max |ax_solver - ax_numerical|: {np.max(ax_err):.6f}")
print(f"Max |ay_solver - ay_numerical|: {np.max(ay_err):.6f}")
print(f"Mean acceleration error: {np.mean(np.sqrt(ax_err**2 + ay_err**2)):.6f}")

## 10. Second-Order Validation: Acceleration from Position

As an additional check, compute acceleration directly as the second
derivative of the position trajectory using `numpy.gradient` twice.

In [ ]:
# Second derivative of position
ax_num2 = np.gradient(np.gradient(cx_kin, dt), dt)
ay_num2 = np.gradient(np.gradient(cy_kin, dt), dt)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(crank_angle_deg, ax_c, 'r-', linewidth=2, label='Solver ax')
axes[0].plot(crank_angle_deg, ax_num2, 'k--', linewidth=1.5, label='d^2(pos)/dt^2')
axes[0].set_ylabel('Acceleration X')
axes[0].set_title('Acceleration validation: solver vs second derivative of position')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(crank_angle_deg, ay_c, 'b-', linewidth=2, label='Solver ay')
axes[1].plot(crank_angle_deg, ay_num2, 'k--', linewidth=1.5, label='d^2(pos)/dt^2')
axes[1].set_xlabel('Crank angle (degrees)')
axes[1].set_ylabel('Acceleration Y')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

ax_err2 = np.abs(ax_c[margin:-margin] - ax_num2[margin:-margin])
ay_err2 = np.abs(ay_c[margin:-margin] - ay_num2[margin:-margin])
print(f"Max |ax_solver - d^2x/dt^2|: {np.max(ax_err2):.6f}")
print(f"Max |ay_solver - d^2y/dt^2|: {np.max(ay_err2):.6f}")
print(f"Mean error: {np.mean(np.sqrt(ax_err2**2 + ay_err2**2)):.6f}")

## 11. Peak Velocity and Acceleration

Peak values are important for dynamic force analysis. Higher accelerations
mean larger inertia forces on the links.

In [ ]:
print("Peak kinematic values for each joint:")
header = (
    f"{'Joint':<8} {'Max speed':>12} {'Max accel':>12} "
    f"{'Angle at max v':>16} {'Angle at max a':>16}"
)
print(header)
print("-" * 68)

joint_names = ['A', 'D', 'B', 'C']
for j_idx in range(min(4, n_joints)):
    jvx = vel_traj[:, j_idx, 0]
    jvy = vel_traj[:, j_idx, 1]
    jv_mag = np.sqrt(jvx**2 + jvy**2)

    jax = acc_traj[:, j_idx, 0]
    jay = acc_traj[:, j_idx, 1]
    ja_mag = np.sqrt(jax**2 + jay**2)

    max_v = np.max(jv_mag)
    max_a = np.max(ja_mag)
    angle_max_v = crank_angle_deg[np.argmax(jv_mag)]
    angle_max_a = crank_angle_deg[np.argmax(ja_mag)]

    name = joint_names[j_idx] if j_idx < len(joint_names) else f"J{j_idx}"
    print(
        f"{name:<8} {max_v:>12.4f} {max_a:>12.4f} "
        f"{angle_max_v:>14.1f} deg {angle_max_a:>14.1f} deg"
    )

In [ ]:
# Velocity and acceleration magnitude for all moving joints in one plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for j_idx in range(n_joints):
    jvx = vel_traj[:, j_idx, 0]
    jvy = vel_traj[:, j_idx, 1]
    jv_mag = np.sqrt(jvx**2 + jvy**2)

    jax = acc_traj[:, j_idx, 0]
    jay = acc_traj[:, j_idx, 1]
    ja_mag = np.sqrt(jax**2 + jay**2)

    if np.max(jv_mag) < 1e-10:
        continue  # Skip static joints

    name = joint_names[j_idx] if j_idx < len(joint_names) else f"J{j_idx}"
    axes[0].plot(crank_angle_deg, jv_mag, linewidth=1.5, label=name)
    axes[1].plot(crank_angle_deg, ja_mag, linewidth=1.5, label=name)

axes[0].set_ylabel('Speed')
axes[0].set_title('Speed of all moving joints')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Crank angle (degrees)')
axes[1].set_ylabel('Acceleration magnitude')
axes[1].set_title('Acceleration magnitude of all moving joints')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the full kinematic analysis pipeline:

1. **Build** a four-bar linkage from link lengths using `fourbar_from_lengths()`.
2. **Convert** to solver arrays with `linkage_to_solver_data()` for numba performance.
3. **Simulate** positions with `simulate()`, or full kinematics with `simulate_with_kinematics()`.
4. **Validate** the analytical velocity and acceleration against finite-difference approximations. Small residual errors are expected since finite differences are O(h^2) approximations while the solver uses exact analytical derivatives.
5. **Analyze** peak velocity and acceleration for each joint, which feeds into dynamic force estimation.

The analytical solver produces velocity and acceleration data that closely matches
numerical differentiation, confirming the correctness of the implicit differentiation
approach used in the revolute joint velocity and acceleration solvers.